In [ ]:
# Centralized imports (StarDist only)
from pathlib import Path
import time
import numpy as np
import pandas as pd
import tifffile
from tifffile import imwrite
from skimage.transform import rescale
import matplotlib.pyplot as plt

from csbdeep.utils import normalize
from stardist.models import StarDist2D, StarDist3D

# StarDist runs on TensorFlow. Native Windows TensorFlow GPU support ended at TF 2.10;
# for GPU acceleration on recent cards use WSL2 / Linux with `tensorflow[and-cuda]`.
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
status = {
    "tf_version": tf.__version__,
    "gpu_available": len(gpus) > 0,
    "gpus": [g.name for g in gpus],
}
print(status)


c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\torch_em\util\image.py:6: UserWarning: elf has switched from the affogato, vigra, and nifty librares to https://github.com/computational-cell-analytics/bioimage-cpp as new backed for custom functionality implemented in C++, e.g. mutex watershed, multicut etc. This may lead to some changes in behavior and interface. If you run into issues with the new version consider installing a version < 0.9. Please also consider raising an issue on github so that we are aware of issues with the migration.
  from elf.io import open_file
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found.

{'cuda_available': True, 'device_count': 1, 'device_name': 'NVIDIA GeForce RTX 5080'}


In [ ]:

_MODEL_CACHE = {}  # cache loaded models across instances so we don't have to reload them for each image


def _iou_matrix(masks_a, masks_b):
    """Label-to-label intersection-over-union matrix between two 2D label images.

    Rows index labels of `masks_a` (0 = background), columns index labels of `masks_b`.
    """
    a = masks_a.ravel().astype(np.int64)
    b = masks_b.ravel().astype(np.int64)
    na = int(a.max()) + 1
    nb = int(b.max()) + 1
    overlap = np.zeros((na, nb), dtype=np.int64)
    np.add.at(overlap, (a, b), 1)
    n_a = overlap.sum(axis=1, keepdims=True)
    n_b = overlap.sum(axis=0, keepdims=True)
    union = n_a + n_b - overlap
    return np.where(union > 0, overlap / union, 0.0)


def _stitch_3d(masks, stitch_threshold=0.25):
    """Stitch a stack of 2D label planes (Z, Y, X) into a 3D labelling by IoU overlap
    between adjacent planes. Standalone reimplementation of the standard stitching used
    by cellpose, so this notebook has no cellpose dependency."""
    masks = masks.astype(np.int32).copy()
    if masks.shape[0] == 0:
        return masks
    mmax = int(masks[0].max())
    for i in range(masks.shape[0] - 1):
        iou = _iou_matrix(masks[i + 1], masks[i])[1:, 1:]  # drop background row/col
        icount = int(masks[i + 1].max())
        if iou.size == 0:
            if icount > 0:
                istitch = np.append(0, np.arange(mmax + 1, mmax + icount + 1))
                masks[i + 1] = istitch[masks[i + 1]]
                mmax += icount
            continue
        iou[iou < stitch_threshold] = 0.0
        iou[iou < iou.max(axis=0)] = 0.0
        istitch = iou.argmax(axis=1) + 1
        ino = np.nonzero(iou.max(axis=1) == 0.0)[0]
        istitch[ino] = np.arange(mmax + 1, mmax + len(ino) + 1)
        mmax += len(ino)
        istitch = np.append(0, istitch)
        masks[i + 1] = istitch[masks[i + 1]]
    return masks


class SegmentationComparisons:
    """Run StarDist nuclear segmentation (and parameter/model sweeps) on the same image
    and compare against an existing/prior segmentation.

    The input image is a 2-channel z-stack stored as (Z, C, Y, X), where
    channel `dapi_channel` is DAPI and channel `bf_channel` is brightfield/phase.
    Every (method, parameter-combo) writes a label mask + overlay PNG to
    `output_dir` and contributes one row to the results table.
    """

    def __init__(self, input_csv, index, scale_factor_xy=3, scale_factor_z=2,
        output_dir=Path(r"Z:\Bel\Jorge_SPACEFISH_Examples\stardist_comparisons"),
        dapi_channel=0, bf_channel=1):
        self.input_csv = input_csv
        self.index = index
        self.scale_factor_xy = float(scale_factor_xy)
        self.scale_factor_z = float(scale_factor_z)

        row = input_csv.loc[index]
        self.two_channel_image_path = Path(row["2_channel_tif_save_path"])
        self.dapi_channel = int(dapi_channel)
        self.bf_channel = int(bf_channel)
        self.image_name = row["image_name"]

        self.output_dir = Path(output_dir)
        self.existing_segmentation = tifffile.imread(Path(row["existing_segmentation_path"]))

        # (Z, C, Y, X)
        self.two_channel_image = tifffile.imread(self.two_channel_image_path)
        if self.two_channel_image.ndim != 4:
            raise ValueError(f"Expected a 4D (Z, C, Y, X) image, got shape {self.two_channel_image.shape}")

        self.results = {}  # label -> label array
        self.counts = {}   # label -> object count

    def pixel_size(self):
        with tifffile.TiffFile(self.two_channel_image_path) as tif:
            tags = {tag.name: tag.value for tag in tif.pages[0].tags.values()}
            x_um = 1 / (tags["XResolution"][0] / tags["XResolution"][1])
            y_um = 1 / (tags["YResolution"][0] / tags["YResolution"][1])
            try:
                z_um = float(str(tags["IJMetadata"]).split("nscales=")[1].split(",")[2].split("\\nunit")[0])
            except Exception:
                z_um = float(str(tags["ImageDescription"]).split("spacing=")[1].split("loop")[0])
        self.original_spacing = (x_um, y_um, z_um)

    def rescale_image(self):
        """Downsample DAPI and brightfield channels and compute z-anisotropy."""
        self.pixel_size()
        x_um, y_um, z_um = self.original_spacing
        xy_ratio = 1.0 / self.scale_factor_xy
        z_ratio = 1.0 / self.scale_factor_z

        dapi = self.two_channel_image[:, self.dapi_channel].astype(np.float32)  # (Z, Y, X)
        bf = self.two_channel_image[:, self.bf_channel].astype(np.float32)

        self.dapi_ds = rescale(dapi, (z_ratio, xy_ratio, xy_ratio), anti_aliasing=True, preserve_range=True).astype(np.float32)
        self.bf_ds = rescale(bf, (z_ratio, xy_ratio, xy_ratio), anti_aliasing=True, preserve_range=True).astype(np.float32)

        # downsample the existing/prior label mask with nearest-neighbour (order=0, no anti-aliasing)
        # so labels are preserved exactly and it stays pixel-aligned with the new segmentations
        self.existing_segmentation_ds = rescale(
            self.existing_segmentation, (z_ratio, xy_ratio, xy_ratio),
            order=0, anti_aliasing=False, preserve_range=True).astype(self.existing_segmentation.dtype)

        # physical voxel spacing after downsampling
        self.z_spacing_ds = z_um * self.scale_factor_z
        self.xy_spacing_ds = x_um * self.scale_factor_xy
        self.anisotropy = self.z_spacing_ds / self.xy_spacing_ds
        self.pixel_size_um_2d = float(self.xy_spacing_ds)
        print(f"downsampled DAPI shape {self.dapi_ds.shape}, anisotropy {self.anisotropy:.3f}")

    # ---- model loaders (cached) ----

    @staticmethod
    def _get_stardist2d(model_name):
        key = f"sd2d_{model_name}"
        if key not in _MODEL_CACHE:
            _MODEL_CACHE[key] = StarDist2D.from_pretrained(model_name)
        return _MODEL_CACHE[key]

    @staticmethod
    def _get_stardist3d(model_name):
        key = f"sd3d_{model_name}"
        if key not in _MODEL_CACHE:
            _MODEL_CACHE[key] = StarDist3D.from_pretrained(model_name)
        return _MODEL_CACHE[key]

    # ---- segmentation methods: each takes its swept params and RETURNS a label volume ----

    def stardist_2d_stitched(self, model_name="2D_versatile_fluo", prob_thresh=None,
                             nms_thresh=None, stitch_threshold=0.1):
        """StarDist 2D applied per-z (DAPI only) then stitched into 3D by IoU overlap.
        `prob_thresh`/`nms_thresh` of None use the model's optimized defaults."""
        model = self._get_stardist2d(model_name)
        z_masks = []
        for z in range(self.dapi_ds.shape[0]):
            plane = normalize(self.dapi_ds[z], 1, 99.8)
            lab, _ = model.predict_instances(plane, prob_thresh=prob_thresh, nms_thresh=nms_thresh)
            z_masks.append(lab.astype(np.int32))
        stacked = np.stack(z_masks, axis=0)
        return _stitch_3d(stacked, stitch_threshold=stitch_threshold)

    def stardist_3d(self, model_name="3D_demo", prob_thresh=None, nms_thresh=None):
        """StarDist native-3D on the DAPI volume. NOTE: the only bundled 3D model is the
        '3D_demo' weights, so results are a baseline rather than production quality."""
        model = self._get_stardist3d(model_name)
        vol = normalize(self.dapi_ds, 1, 99.8)
        labels, _ = model.predict_instances(vol, prob_thresh=prob_thresh, nms_thresh=nms_thresh)
        return labels

    # ---- output helpers ----

    @staticmethod
    def _filter_min_size(labels, min_size):
        """Remove objects smaller than `min_size` voxels (avoids counting debris)."""
        if not min_size:
            return labels
        ids, counts = np.unique(labels, return_counts=True)
        remove = ids[(ids != 0) & (counts < min_size)]
        if remove.size:
            labels = labels.copy()
            labels[np.isin(labels, remove)] = 0
        return labels

    @staticmethod
    def _param_label(params):
        """Short filesystem-safe string describing a parameter combo."""
        parts = [f"{k}={v}" for k, v in params.items()]
        label = "__".join(parts) if parts else "default"
        return label.replace(".", "p").replace("-", "neg")

    def _save_overlay_png(self, seg, png_path, method_name, suptitle):
        """Save a max-projection figure with four panels:
        DAPI, brightfield, the NEW StarDist labels over brightfield, and the
        EXISTING (prior) segmentation over brightfield (both pixel-aligned).
        """
        dapi_mip = self.dapi_ds.max(axis=0)
        bf_mip = self.bf_ds.max(axis=0)
        new_label_mip = seg.max(axis=0) if seg.ndim == 3 else seg

        # existing/prior segmentation, downsampled to match the new segmentations (pixel-aligned)
        existing = self.existing_segmentation_ds
        existing_mip = existing.max(axis=0) if existing.ndim == 3 else existing

        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        axes[0].imshow(dapi_mip, cmap="gray")
        axes[0].set_title("DAPI (max projection)")
        axes[1].imshow(bf_mip, cmap="gray")
        axes[1].set_title("Brightfield (max projection)")

        axes[2].imshow(bf_mip, cmap="gray")
        new_overlay = np.ma.masked_where(new_label_mip == 0, new_label_mip)
        axes[2].imshow(new_overlay, cmap="nipy_spectral", alpha=0.5, interpolation="nearest")
        axes[2].set_title(f"NEW method: {method_name}")

        axes[3].imshow(bf_mip, cmap="gray")
        existing_overlay = np.ma.masked_where(existing_mip == 0, existing_mip)
        axes[3].imshow(existing_overlay, cmap="nipy_spectral", alpha=0.5, interpolation="nearest")
        axes[3].set_title("EXISTING (prior) segmentation")

        for ax in axes:
            ax.axis("off")
        fig.suptitle(suptitle)
        fig.tight_layout()
        fig.savefig(png_path, dpi=150, bbox_inches="tight")
        plt.close(fig)

    def _build_jobs(self, stardist2d_models, stardist3d_models, prob_threshs,
                    nms_threshs, stitch_thresholds, include):
        """Expand the model/parameter grids into a flat list of (method, params, fn) jobs."""
        jobs = []
        if "stardist_2dstitch" in include:
            for model_name in stardist2d_models:
                for stitch in stitch_thresholds:
                    for pt in prob_threshs:
                        for nt in nms_threshs:
                            jobs.append(("stardist_2dstitch",
                                         {"model_name": model_name, "stitch_threshold": stitch,
                                          "prob_thresh": pt, "nms_thresh": nt},
                                         self.stardist_2d_stitched))
        if "stardist_3d" in include:
            for model_name in stardist3d_models:
                for pt in prob_threshs:
                    for nt in nms_threshs:
                        jobs.append(("stardist_3d",
                                     {"model_name": model_name, "prob_thresh": pt, "nms_thresh": nt},
                                     self.stardist_3d))
        return jobs

    ######### MAIN PART############
    def run_sweep(self,
                  stardist2d_models=("2D_versatile_fluo", "2D_paper_dsb2018"),
                  stardist3d_models=("3D_demo",),
                  prob_threshs=(None,),
                  nms_threshs=(None,),
                  stitch_thresholds=(0.1, 0.25),
                  min_size=500,
                  include=("stardist_2dstitch",)):
        """Run every (model, parameter-combo). Saves a TIF + overlay PNG per combo and
        returns one results row per successful run.

        Tweak `stardist2d_models` / `prob_threshs` / `nms_threshs` / `stitch_thresholds`
        to explore models and thresholds. Add "stardist_3d" to `include` to also run the
        native-3D demo model."""
        self.rescale_image()
        jobs = self._build_jobs(stardist2d_models, stardist3d_models, prob_threshs,
                                nms_threshs, stitch_thresholds, include)
        print(f"{self.image_name}: {len(jobs)} jobs queued")

        self.run_results = []
        for method_name, params, fn in jobs:
            label = f"{method_name}__{self._param_label(params)}"
            try:
                t0 = time.time()
                seg = fn(**params)
                seg = self._filter_min_size(seg, min_size)
                time_taken = time.time() - t0

                count = int(np.unique(seg).size - (1 if (seg == 0).any() else 0))
                self.counts[label] = count
                self.results[label] = seg
                print(f"  {label}: {count} objects, {time_taken:.1f}s")

                method_dir = self.output_dir / method_name
                method_dir.mkdir(parents=True, exist_ok=True)
                stem = f"{self.image_name}__{label}"
                imwrite(method_dir / f"{stem}.tif", seg.astype(np.uint32))
                self._save_overlay_png(seg, method_dir / f"{stem}.png", method_name, stem)

                row = {"image_name": self.image_name, "method": method_name,
                       "time_taken": time_taken, "objects_found": count, "min_size": min_size}
                row.update(params)
                self.run_results.append(row)
            except Exception as exc:
                print(f"[SKIP] {label}: {type(exc).__name__}: {exc}")

        return self.run_results


In [3]:
input_csv = pd.read_excel(r"z:\Bel\Jorge_SPACEFISH_Examples\image_locations.xlsx")
input_csv.head()


,image_name,path,dapi_channel,bf_channel,image_type,scene_id,"censor region (z1,z2,x1_z1, x2_z1, y1_z1,y2_z1,x1_z2,x2_z2,y1_z2,y2_z2)",existing_segmentation_path,2_channel_tif_save_path
0,dev4_1_6h,Z:\Jorge\20241124_6h_dev4\20241124_dev4_6h_mer...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
1,dev4_2_6h,Z:\Jorge\20241125_repeats_6h_2d_pin255\2024112...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
2,dev4_3_6h,Z:\Jorge\20241124_6h_dev4\20241124_dev4_6h_mer...,6,7,vascu,2,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
3,dev7_3_day1,Z:\Jorge\20241126_day1_dev7\20241126_dev7_day1...,6,7,vascu,0,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...
4,dev7_2_day1,Z:\Jorge\20241126_day1_dev7\20241126_dev7_day1...,6,7,vascu,1,NaN,Z:\Jorge\SPACEFISH_analysis\2026\v115-vascu-01...,Z:\Bel\Jorge_SPACEFISH_Examples\two_channel_im...


In [ ]:
# run the StarDist sweep on every image, collecting one results row per successful run
# save a per-image CSV as soon as each image finishes so an early termination doesn't lose data
all_results = []
for index in input_csv.index:
    comparison = SegmentationComparisons(index=index, input_csv=input_csv, scale_factor_xy=3, scale_factor_z=2)
    image_results = comparison.run_sweep()
    all_results.extend(image_results)

    # write this image's results immediately
    image_df = pd.DataFrame(image_results)
    safe_name = "".join(c if c.isalnum() or c in "-_." else "_" for c in str(comparison.image_name))
    image_csv_path = comparison.output_dir / f"results_{safe_name}.csv"
    image_df.to_csv(image_csv_path, index=False)
    print(f"saved {len(image_df)} rows -> {image_csv_path}")

# single combined CSV across all images, models, and parameter combos
results_df = pd.DataFrame(all_results)
results_csv_path = comparison.output_dir / "stardist_comparison_results.csv"
results_df.to_csv(results_csv_path, index=False)
print(f"saved {len(results_df)} rows -> {results_csv_path}")
results_df

# to explore different models / thresholds, pass narrower or wider grids, e.g.:
# comparison.run_sweep(stardist2d_models=("2D_versatile_fluo",), prob_threshs=(0.4, 0.5))
# to also run the native-3D demo model:
# comparison.run_sweep(include=("stardist_2dstitch", "stardist_3d"))
# to run a single image:
# comparison = SegmentationComparisons(index=0, input_csv=input_csv)
# comparison.run_sweep()


INFO:cellpose.core:** TORCH CUDA version installed and working. **
INFO:cellpose.core:>>>> using GPU (CUDA)


downsampled DAPI shape (63, 1302, 1311), anisotropy 2.838
dev4_1_6h: 21 jobs queued


INFO:cellpose.models:>>>> loading model C:\Users\taylorhearn\.cellpose\models\cpsam
INFO:cellpose.core:100%|##########| 63/63 [01:13<00:00,  1.16s/it]
INFO:cellpose.models:network run in 73.02s
INFO:cellpose.models:0%|          | 0/63 [00:00<?, ?it/s]
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\cellpose_napari\lib\site-packages\cellpose\dynamics.py:524: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  coo = torch.sparse_coo_tensor(pt, torch.ones(pt.shape[1], device=pt.device, dtype=torch.int),
INFO:cellpose.models:100%|##########| 63/63 [00:23<00:00,  2.65it/s]
INFO:cellpose.models:stitching 63 pl

  cpsam_2dstitch__channels=dapi_bf__stitch_threshold=0p1__cellprob_threshold=neg1p0__flow_threshold=0p4__min_size=500: 1601 objects, 118.0s


INFO:cellpose.core:100%|##########| 63/63 [01:13<00:00,  1.17s/it]
INFO:cellpose.models:network run in 73.46s
INFO:cellpose.models:100%|##########| 63/63 [00:21<00:00,  2.94it/s]
INFO:cellpose.models:stitching 63 planes using stitch_threshold=0.100 to make 3D masks
100%|██████████| 62/62 [00:05<00:00, 11.36it/s]
INFO:cellpose.models:masks created in 28.49s
